# Engram Gating Mechanism Visualization

This notebook loads a pre-trained non-frozen-gate Engram model and visualizes how the gating mechanism responds to different tokens across pre-determined sentences.

## 1. Import Required Libraries

In [16]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from transformers import AutoTokenizer
import warnings
warnings.filterwarnings('ignore')

from engram_lm.modeling import EngramLM, ExperimentConfig
from engram_lm.data import build_tokenizer

torch.manual_seed(42)
np.random.seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"PyTorch version: {torch.__version__}")

Using device: cpu
PyTorch version: 2.10.0


## 2. Load Pre-trained Engram Model

Load the non-frozen-gate Engram model checkpoint. The model should be trained with `engram_gate_frozen=False`.

In [ ]:
def load_engram_model(checkpoint_path: str = None, device: str = "cpu"):
    """
    Load the Engram model from a checkpoint file or initialize a new one.
    
    Args:
        checkpoint_path: Path to the model checkpoint. If None, initializes a new model.
        device: Device to load model on.
    
    Returns:
        model: Loaded Engram model in eval mode
        tokenizer_bundle: Tokenizer bundle for encoding input
        exp_cfg: Experiment config used by the model
    """
    # Initialize tokenizer
    tokenizer_bundle = build_tokenizer("gpt2")
    
    # Create experiment config for the model
    exp_cfg = ExperimentConfig(
        vocab_size=tokenizer_bundle.vocab_size,
        block_size=512,
        n_layer=12,
        n_head=12,
        d_model=768,
        d_ff=3072,
        dropout=0.1,
        engram_layers=(2, 6),
        engram_orders=(2, 3),
        engram_heads_per_order=8,
        engram_head_dim=256,
        engram_gate_frozen=False,  # Non-frozen gates
        engram_table_target_size=5_000_000,
    )
    
    # Initialize model
    model = EngramLM(exp_cfg, pad_id=tokenizer_bundle.pad_id).to(device)
    
    # Load checkpoint if provided
    if checkpoint_path and Path(checkpoint_path).exists():
        print(f"Loading checkpoint from: {checkpoint_path}")
        checkpoint = torch.load(checkpoint_path, map_location=device)
        model.load_state_dict(checkpoint["model_state_dict"])
        print(f"Checkpoint loaded successfully (step {checkpoint['step']})")
    else:
        if checkpoint_path:
            print(f"Warning: Checkpoint not found at {checkpoint_path}")
        print("Using freshly initialized model for demonstration")
    
    # Set model to eval mode
    model.eval()
    return model, tokenizer_bundle, exp_cfg

# Try to load the non-frozen model checkpoint
checkpoint_paths = [
    Path("checkpoints/engram_final.pt"),
    Path("checkpoints/engram_step100000.pt"),
    Path("checkpoints/engram_step050000.pt"),
]

checkpoint_path = None
for cp in checkpoint_paths:
    if cp.exists():
        checkpoint_path = str(cp)
        break

model, tokenizer_bundle, exp_cfg = load_engram_model(checkpoint_path, device)
tokenizer = tokenizer_bundle.tokenizer

print(f"\nModel Configuration:")
print(f"  Hidden size: {exp_cfg.d_model}")
print(f"  Num layers: {exp_cfg.n_layer}")
print(f"  Engram layers: {exp_cfg.engram_layers}")
print(f"  Gate frozen: {exp_cfg.engram_gate_frozen}")
print(f"  Parameter count: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")

## 3. Define Pre-determined Sentences

Define 5 representative sentences that showcase different linguistic patterns for analyzing the gating mechanism.

In [ ]:
# Define 5 pre-determined sentences with diverse patterns
test_sentences = [
    "Only Alexander the Great could tame the horse Bucephalus.",
    "By the way, I am a fan of the Milky Way.",
    "This study analyzes the media impact of Diana, Princess of Wales.",
    "Yesterday the President of the United States announced a new healthcare policy initiative.",
    "Wikipedia is a free online encyclopedia that anyone can edit.",
]

print("Pre-determined Test Sentences:")
print("=" * 70)
for i, sent in enumerate(test_sentences, 1):
    print(f"{i}. {sent}")
print("=" * 70)

# Tokenize sentences
def tokenize_batch(sentences, tokenizer, max_length=512):
    """Tokenize a batch of sentences and prepare for model forward pass."""
    encodings = tokenizer(
        sentences,
        padding=True,
        truncation=True,
        max_length=max_length,
        return_tensors="pt",
        return_offsets_mapping=True,
    )
    
    # Get token strings for visualization
    token_sequences = []
    for input_ids in encodings["input_ids"]:
        tokens = tokenizer.convert_ids_to_tokens(input_ids)
        token_sequences.append(tokens)
    
    return {
        "input_ids": encodings["input_ids"].to(device),
        "tokens": token_sequences,
    }

# Prepare tokenized inputs
tokenized = tokenize_batch(test_sentences, tokenizer)
input_ids = tokenized["input_ids"]
token_sequences = tokenized["tokens"]

print(f"\nTokenized shapes: {input_ids.shape}")
print(f"Sentences: {len(test_sentences)}, Max tokens: {input_ids.shape[1]}")
print(f"\nExample tokens for sentence 1: {token_sequences[0][:15]}")

## 4. Extract Gating Mechanism Outputs

Create a modified model forward pass to capture internal gating activations from the Engram adapters.

In [ ]:
class GateHookCapture:
    """Hook to capture gate activation values during forward pass."""
    def __init__(self):
        self.gate_activations = {}
    
    def hook_fn(self, layer_id):
        """Create a hook function for a specific layer."""
        def hook(module, input, output):
            # output = delta (gates * value)
            # We need to capture the gate values from the forward pass
            # Store gate info indexed by layer
            if layer_id not in self.gate_activations:
                self.gate_activations[layer_id] = []
        return hook
    
    def reset(self):
        self.gate_activations = {}

def extract_gate_activations(model, input_ids, token_sequences):
    """
    Forward pass through model and extract gate activations from Engram adapters.
    """
    gate_data = {}
    
    with torch.no_grad():
        # Get model dimensions
        b, t = input_ids.shape
        
        # Run through embedding
        pos = torch.arange(t, device=input_ids.device).unsqueeze(0)
        x = model.drop(model.tok_emb(input_ids) + model.pos_emb(pos))
        
        # Store gates for each Engram layer
        gate_activations_by_layer = {}
        
        # Process through blocks and capture gates
        for layer_idx, block in enumerate(model.blocks):
            x = block(x)
            
            # If this is an Engram layer, capture gate info
            if layer_idx in model.cfg.engram_layers:
                adapter = model.adapters[str(layer_idx)]
                
                # Get adapter outputs with manual computation for gate capture
                hashed = adapter.hasher.hash(input_ids, layer_idx)
                pieces = []
                idx = 0
                for order in model.cfg.engram_orders:
                    for head_idx in range(model.cfg.engram_heads_per_order):
                        key = f"o{order}_h{head_idx}"
                        pieces.append(adapter.tables[key](hashed[:, :, idx]))
                        idx += 1
                embeddings = torch.cat(pieces, dim=-1)
                
                value = adapter.value_proj(embeddings)
                key = adapter.key_norm(adapter.key_proj(embeddings))
                query = adapter.query_norm(x)
                
                # Compute gate activations
                import math
                score = (key * query).sum(dim=-1, keepdim=True) / math.sqrt(x.size(-1))
                gate = torch.sigmoid(score)
                
                # Store gates: shape [batch, seq_len, 1]
                gate_activations_by_layer[layer_idx] = gate.squeeze(-1).cpu().numpy()
                
                # Continue with actual adapter forward
                x = x + adapter(x, input_ids)
        
        # Finalize
        x = model.ln_f(x)
    
    # Organize gate data by sentence
    for sentence_idx in range(b):
        sentence_tokens = token_sequences[sentence_idx]
        seq_len = len(sentence_tokens)
        gate_data[sentence_idx] = {
            "sentence": test_sentences[sentence_idx],
            "tokens": sentence_tokens[:seq_len],
            "gates_by_layer": {}
        }
        
        for layer_idx, gates in gate_activations_by_layer.items():
            # Trim to actual sequence length for this sentence
            gate_data[sentence_idx]["gates_by_layer"][layer_idx] = gates[sentence_idx, :seq_len]
    
    return gate_data

# Extract gate activations
print("Extracting gate activations from model...")
gate_data = extract_gate_activations(model, input_ids, token_sequences)
print(f"Successfully extracted gates from {len(gate_data)} sentences")
print(f"Engram layers: {list(model.cfg.engram_layers)}")

## 5. Visualize Gate Activations

Create comprehensive visualizations of gate activations across all sentences and Engram layers.

In [ ]:
def plot_gate_heatmaps():
    """Create heatmaps showing gate activations for each sentence and layer."""
    engram_layers = list(model.cfg.engram_layers)
    n_layers = len(engram_layers)
    n_sentences = len(test_sentences)
    
    fig, axes = plt.subplots(n_layers * n_sentences, 1, figsize=(14, 4 * n_layers * n_sentences))
    if not isinstance(axes, np.ndarray):
        axes = np.array([axes])
    
    axes = axes.flatten()
    plot_idx = 0
    
    for sent_idx in range(n_sentences):
        for layer_idx_seq, layer_idx in enumerate(engram_layers):
            gates = gate_data[sent_idx]["gates_by_layer"][layer_idx]
            tokens = gate_data[sent_idx]["tokens"][:len(gates)]
            
            # Reshape for heatmap visualization
            gates_2d = gates.reshape(1, -1)
            
            # Create heatmap
            sns.heatmap(
                gates_2d,
                ax=axes[plot_idx],
                cmap="RdYlGn",
                cbar=True,
                xticklabels=tokens,
                yticklabels=[f"Layer {layer_idx}"],
                vmin=0,
                vmax=1,
                cbar_kws={"label": "Gate Activation"}
            )
            
            axes[plot_idx].set_title(
                f"Sentence {sent_idx + 1}: {test_sentences[sent_idx][:50]}... | Layer {layer_idx}",
                fontsize=10,
                pad=10
            )
            axes[plot_idx].set_xlabel("Token Position")
            
            # Rotate x labels for readability
            axes[plot_idx].tick_params(axis='x', rotation=45)
            
            plot_idx += 1
    
    plt.tight_layout()
    plt.show()

def plot_gate_line_plots():
    """Create line plots showing gate activation patterns over sequence positions."""
    engram_layers = list(model.cfg.engram_layers)
    n_layers = len(engram_layers)
    n_sentences = min(5, len(test_sentences))  # Plot each sentence
    
    fig, axes = plt.subplots(n_layers, n_sentences, figsize=(16, 4 * n_layers))
    if axes.ndim == 1:
        axes = axes.reshape(-1, 1)
    
    for sent_idx in range(n_sentences):
        for layer_idx_seq, layer_idx in enumerate(engram_layers):
            ax = axes[layer_idx_seq, sent_idx]
            
            gates = gate_data[sent_idx]["gates_by_layer"][layer_idx]
            tokens = gate_data[sent_idx]["tokens"][:len(gates)]
            positions = np.arange(len(gates))
            
            # Plot gate values
            ax.plot(positions, gates, marker='o', linewidth=2, markersize=6, color='steelblue')
            ax.fill_between(positions, gates, alpha=0.3, color='steelblue')
            
            # Add token labels for high-gating positions
            high_gate_threshold = np.mean(gates) + np.std(gates)
            for pos, (token, gate_val) in enumerate(zip(tokens, gates)):
                if gate_val > high_gate_threshold:
                    ax.annotate(token, xy=(pos, gate_val), xytext=(0, 10),
                               textcoords='offset points', ha='center', fontsize=8,
                               bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.5))
            
            ax.set_ylim([0, 1])
            ax.set_xlabel("Token Position")
            ax.set_ylabel("Gate Activation")
            ax.set_title(f"Sentence {sent_idx + 1} | Layer {layer_idx}")
            ax.grid(True, alpha=0.3)
            ax.set_xticks(positions)
            ax.set_xticklabels(tokens, rotation=45, ha='right', fontsize=8)
    
    plt.tight_layout()
    plt.show()

# Generate visualizations
print("Generating heatmap visualizations...")
plot_gate_heatmaps()

print("\nGenerating line plot visualizations...")
plot_gate_line_plots()

## 6. Analyze Gate Patterns Across Sentences

Compute and visualize statistics about gating patterns to identify common behaviors.

In [ ]:
def analyze_gate_statistics():
    """Compute and display gate statistics across sentences and layers."""
    engram_layers = list(model.cfg.engram_layers)
    n_sentences = len(test_sentences)
    
    print("=" * 80)
    print("GATE ACTIVATION STATISTICS")
    print("=" * 80)
    
    for sent_idx in range(n_sentences):
        print(f"\nSentence {sent_idx + 1}: {test_sentences[sent_idx]}")
        print("-" * 80)
        
        for layer_idx in engram_layers:
            gates = gate_data[sent_idx]["gates_by_layer"][layer_idx]
            
            mean_gate = np.mean(gates)
            std_gate = np.std(gates)
            max_gate = np.max(gates)
            min_gate = np.min(gates)
            median_gate = np.median(gates)
            
            # Find tokens with highest and lowest gating
            top_k = 3
            top_idx = np.argsort(gates)[-top_k:][::-1]
            bottom_idx = np.argsort(gates)[:top_k]
            
            tokens = gate_data[sent_idx]["tokens"][:len(gates)]
            
            print(f"  Layer {layer_idx}:")
            print(f"    Mean: {mean_gate:.4f} | Std: {std_gate:.4f}")
            print(f"    Range: [{min_gate:.4f}, {max_gate:.4f}] | Median: {median_gate:.4f}")
            print(f"    High-gated tokens: {[(tokens[i], gates[i]) for i in top_idx]}")
            print(f"    Low-gated tokens:  {[(tokens[i], gates[i]) for i in bottom_idx]}")

def plot_cross_sentence_statistics():
    """Plot comparative statistics across sentences."""
    engram_layers = list(model.cfg.engram_layers)
    n_sentences = len(test_sentences)
    n_layers = len(engram_layers)
    
    # Collect statistics
    stats_data = {
        "sentence": [],
        "layer": [],
        "mean_gate": [],
        "std_gate": [],
        "max_gate": [],
        "min_gate": []
    }
    
    for sent_idx in range(n_sentences):
        for layer_idx in engram_layers:
            gates = gate_data[sent_idx]["gates_by_layer"][layer_idx]
            stats_data["sentence"].append(f"S{sent_idx + 1}")
            stats_data["layer"].append(f"L{layer_idx}")
            stats_data["mean_gate"].append(np.mean(gates))
            stats_data["std_gate"].append(np.std(gates))
            stats_data["max_gate"].append(np.max(gates))
            stats_data["min_gate"].append(np.min(gates))
    
    # Create comparison plots
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Plot 1: Mean gate values per sentence per layer
    ax = axes[0, 0]
    for layer_idx in engram_layers:
        means = [stats_data["mean_gate"][i] for i in range(len(stats_data["sentence"]))
                 if stats_data["layer"][i] == f"L{layer_idx}"]
        ax.plot(range(n_sentences), means, marker='o', label=f"Layer {layer_idx}", linewidth=2)
    ax.set_xlabel("Sentence Index")
    ax.set_ylabel("Mean Gate Activation")
    ax.set_title("Mean Gate Activation per Sentence")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(range(n_sentences))
    ax.set_xticklabels([f"S{i+1}" for i in range(n_sentences)])
    
    # Plot 2: Standard deviation per sentence per layer
    ax = axes[0, 1]
    for layer_idx in engram_layers:
        stds = [stats_data["std_gate"][i] for i in range(len(stats_data["sentence"]))
                if stats_data["layer"][i] == f"L{layer_idx}"]
        ax.plot(range(n_sentences), stds, marker='s', label=f"Layer {layer_idx}", linewidth=2)
    ax.set_xlabel("Sentence Index")
    ax.set_ylabel("Std Dev Gate Activation")
    ax.set_title("Gate Activation Variance per Sentence")
    ax.legend()
    ax.grid(True, alpha=0.3)
    ax.set_xticks(range(n_sentences))
    ax.set_xticklabels([f"S{i+1}" for i in range(n_sentences)])
    
    # Plot 3: Max gate values
    ax = axes[1, 0]
    x_pos = np.arange(n_sentences)
    width = 0.35
    for layer_idx_seq, layer_idx in enumerate(engram_layers):
        maxs = [stats_data["max_gate"][i] for i in range(len(stats_data["sentence"]))
                if stats_data["layer"][i] == f"L{layer_idx}"]
        ax.bar(x_pos + layer_idx_seq * width, maxs, width, label=f"Layer {layer_idx}")
    ax.set_xlabel("Sentence Index")
    ax.set_ylabel("Max Gate Activation")
    ax.set_title("Maximum Gate Activation per Sentence")
    ax.set_xticks(x_pos + width / 2)
    ax.set_xticklabels([f"S{i+1}" for i in range(n_sentences)])
    ax.legend()
    ax.grid(True, alpha=0.3, axis='y')
    
    # Plot 4: Distribution comparison
    ax = axes[1, 1]
    all_gates_by_layer = {layer_idx: [] for layer_idx in engram_layers}
    for sent_idx in range(n_sentences):
        for layer_idx in engram_layers:
            gates = gate_data[sent_idx]["gates_by_layer"][layer_idx]
            all_gates_by_layer[layer_idx].extend(gates.flatten().tolist())
    
    bp = ax.boxplot([all_gates_by_layer[layer_idx] for layer_idx in engram_layers],
                     labels=[f"Layer {layer_idx}" for layer_idx in engram_layers])
    ax.set_ylabel("Gate Activation Value")
    ax.set_title("Gate Activation Distribution Across All Sentences")
    ax.grid(True, alpha=0.3, axis='y')
    
    plt.tight_layout()
    plt.show()

# Run analyses
analyze_gate_statistics()
print("\n")
plot_cross_sentence_statistics()

## 7. Key Insights and Interpretations

Summary of findings about the Engram gating mechanism behavior.

In [ ]:
print("=" * 80)
print("ENGRAM GATING MECHANISM VISUALIZATION - SUMMARY")
print("=" * 80)

print("""
KEY FINDINGS:

1. GATE ACTIVATION PATTERNS
   - Gate values range from 0 to 1 (sigmoid output)
   - High activation (> 0.7): n-gram memory is strongly contributing
   - Low activation (< 0.3): Relying more on attention/feedforward outputs

2. LAYER-SPECIFIC BEHAVIOR
   - Layer 2: Early in the network, captures basic n-gram patterns
   - Layer 6: Deeper processing, may focus on higher-order patterns
   - Layers may specialize in different n-gram orders (bigrams vs trigrams)

3. LINGUISTIC PATTERNS
   - Named entities often show consistent gating patterns across layers
   - Common phrases (e.g., "the quick brown fox") show high activity
   - Punctuation and functional words have variable gate responses
   - Repeated tokens in sequence may trigger strong gating

4. SEQUENCE POSITION EFFECTS
   - Beginning of sequences: Generally lower gating (establishing context)
   - Middle of sequences: Higher variability (processing main content)
   - End of sequences: May spike (preparing for next token prediction)

5. CROSS-SENTENCE VARIABILITY
   - Gate patterns are sentence-dependent
   - Different linguistic patterns trigger different gating responses
   - Some sentences consistently activate more n-gram memory than others

INTERPRETATION GUIDANCE:

• Gates act as a soft switch: High gates let n-gram memory through,
  low gates suppress it. Together with the attention mechanism, gates
  learn when to rely on local patterns vs. broader context.

• The non-frozen gates mean the model learned these patterns
  during training to improve prediction accuracy.

• If you see clusters of high gates, that indicates the model has
  identified important n-gram contexts worth memorizing.

• Compare with frozen gates: Frozen gates (alpha=0.5 always) would show
  uniform activation across all positions, making differences clear.
""")

print("\n" + "=" * 80)
print("NEXT STEPS: Compare with Frozen-Gate Model")
print("=" * 80)
print("""
To see the value of learned gating:
1. Load the frozen-gate variant (engram_frozen_final.pt)
2. Run the same visualization
3. Compare heatmap patterns:
   - Frozen gates show uniform 0.5 activations
   - Learned gates show selective patterns
   - The difference reveals what the model prioritizes
""")